# Machine Learning Modeling, Evaluation & MLflow Tracking

## Project
**Mathematical Fingerprint of Musical Success**

## Objectives
- Train a **Linear Regression** model to predict song popularity.
- Train a **Classification** model (Random Forest) to predict hit status.
- Use **MLflow** for experiment tracking and model versioning.
- Perform **Feature Importance** analysis to decode the "formula for success".
- Evaluate models using advanced metrics (Confusion Matrix, F1-score, RMSE).

---

## Data Setup (Required)
1. Open the **Files** pane (folder icon on the left).
2. Ensure the folder named `data` exists.
3. **Upload** the final processed file from ***math_music_eda_feature_engineering.ipynb***:
   - `data/master_dataset_final.csv`

In [ ]:
# ============================================================
# Topic: Environment Setup
# Goal: Install MLflow and import necessary ML libraries
# ============================================================

# Install MLflow (must be done every session in Colab)
!pip install mlflow -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, mean_absolute_error,
                             mean_squared_error, r2_score)

# experiment Tracking
import mlflow

print("Libraries and MLflow installed successfully.")

In [ ]:
# ==============================================================================
# Topic: Data Loading
# Goal: Load the processed dataset from math_music_eda_feature_engineering.ipynb
# ==============================================================================
import os

processed_file = "data/master_dataset_final.csv"

if os.path.exists(processed_file):
    df_model = pd.read_csv(processed_file)
    print(f"Data loaded successfully: {df_model.shape[0]} rows, {df_model.shape[1]} columns.")

    # quick sanity check for the newly added features from math_music_eda_feature_engineering.ipynb
    required_new_cols = ['energy_loudness_ratio', 'mood_index', 'cluster', 'PC1', 'PC2']
    for col in required_new_cols:
        if col in df_model.columns:
            print(f"   - Feature '{col}' found")
        else:
            print(f"   - Feature '{col}' MISSING! Check math_music_eda_feature_engineering.ipynb output.")

    display(df_model.head(3))
else:
    print(f"ERROR: {processed_file} not found in 'data/' folder. Please upload it.")

In [ ]:
# ============================================================
# Topic: Data Preparation
# Goal: Define feature columns and target variables for
#       classification and regression
# ============================================================

# ------------------------------------------------------------
# Feature selection for modeling
# Keep only numerical / engineered music-related features
# Exclude text/id columns and leakage-prone columns
# ------------------------------------------------------------

feature_cols = [
    "duration_ms",
    "explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature",
    "energy_loudness_ratio",
    "mood_index",
    "PC1",
    "PC2",
    "cluster"
]

# ------------------------------------------------------------
# Convert boolean column to integer if needed
# ------------------------------------------------------------
if df_model["explicit"].dtype == "bool":
    df_model["explicit"] = df_model["explicit"].astype(int)

# ------------------------------------------------------------
# define X and targets
# classification target: is_hit
# regression target: popularity
# ------------------------------------------------------------
X = df_model[feature_cols]

y_class = df_model["is_hit"]
y_reg = df_model["popularity"]

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------
print("Feature matrix and targets created successfully.")
print(f"Number of features: {len(feature_cols)}")
print(f"Feature matrix shape: {X.shape}")
print(f"Classification target shape: {y_class.shape}")
print(f"Regression target shape: {y_reg.shape}")

print("\nSelected features:")
print(feature_cols)

print("\nTarget distribution (is_hit):")
print(y_class.value_counts())

print("\nPopularity summary:")
display(y_reg.describe())

In [ ]:
# mount Google Drive to the Colab environment
# this allows access to files stored in personal Google Drive folders
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
 ============================================================
# Topic: Data Splitting
# Goal:
#   1. Divide the dataset into Training (80%) and Testing (20%) sets
#   2. Use 'stratify' for classification to maintain the rare hit ratio (2.6%)
#   3. Ensure reproducibility using a fixed random_state
# ============================================================

from sklearn.model_selection import train_test_split

# splitting for classification
# goal: train the model on 80% of the songs, and on the remaining 20% ​check whether it recognizes the hits
# use 'stratify' to avoid having no hits in the test due to their rarity
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# splitting for regression
# objective: Preparing data to predict the numerical value of 'popularity'
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print("Data splitting complete.")
print(f"Classification Train/Test: {len(X_train_cls)} / {len(X_test_cls)}")
print(f"Regression Train/Test:     {len(X_train_reg)} / {len(X_test_reg)}")

# check hit ratio consistency
hit_ratio_test = (y_test_cls.sum() / len(y_test_cls)) * 100
print(f"Hit ratio in Test set: {hit_ratio_test:.2f}% (Consistent with master data)")

In [ ]:
# ============================================================
# Topic: Regression Model & MLflow Tracking
# Goal:
#   1. Train a 'Linear Regression' baseline model to predict popularity
#   2. Track the experiment using MLflow (versioning and logging)
#   3. Evaluate the model using MAE, RMSE, and R2 metrics
# ============================================================

# MLflow setup
# goal: create a "log" of the experiment
# If change the model later, MLflow will allow us to compare the results
experiment_name = "Musical_Success_Regression"
try:
    mlflow.create_experiment(experiment_name)
except:
    pass
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="Baseline_Linear_Regression"):
    # target: popularity (0-100)
    # Objective: to test whether musical characteristics explain the state of popularity
    lr_model = LinearRegression()
    lr_model.fit(X_train_reg, y_train_reg)

    # Predictions
    y_pred_reg = lr_model.predict(X_test_reg)

    # Metrics calculation
    mae = mean_absolute_error(y_test_reg, y_pred_reg)
    mse = mean_squared_error(y_test_reg, y_pred_reg)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_reg, y_pred_reg)

    # logging results (MLflow integration)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2_Score", r2)

    print("📊 Linear Regression Results (Popularity Prediction):")
    print("-" * 50)
    print(f"   MAE (Mean Absolute Error): {mae:.2f} points")
    print(f"   RMSE (Root Mean Sq Error): {rmse:.2f} points")
    print(f"   R2 Score (Variance explained): {r2:.4f}")

    print("\n Results successfully logged to MLflow UI.")

### Observations: Linear Regression (Popularity Prediction)

The baseline Linear Regression model achieved an R2 score of only **0.0628**, meaning
that audio features alone explain less than **7% of the variance** in song popularity.

This is a **non-trivial and scientifically meaningful result**:

- It confirms that musical popularity is **not linearly predictable** from audio features.
- A song's success depends on factors **beyond its sound** — marketing, timing, artist
  fame, social media trends, and cultural context all play a role.
- The MAE of ~15 points on a 0-100 scale means the model's predictions are
  consistently "in the right ballpark" but far from precise.

**Conclusion:** Linear Regression provides a starting point for comparison. A more advanced model (Random Forest) is now tested to see if it can predict song popularity more accurately by capturing more complex patterns in the data.

In [ ]:
# ============================================================
# Topic: Classification Model & MLflow Tracking
# Goal:
#   1. Train a Random Forest Classifier to predict hit status (is_hit)
#   2. Handle class imbalance using class_weight='balanced'
#      (Only 2.6% of songs are hits — without balancing, the model
#       would simply predict "not a hit" for everything.)
#   3. Log the experiment to MLflow for comparison with future models
#   4. Evaluate using F1-score (more meaningful than accuracy for imbalanced data)
# ============================================================

experiment_name = "Musical_Success_Classification"
try:
    mlflow.create_experiment(experiment_name)
except:
    pass
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="RandomForest_Balanced_Classification"):

    # train Random Forest
    # class_weight='balanced' automatically gives more weight to rare hits
    rf_model = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train_cls, y_train_cls)

    # predictions
    y_pred_cls = rf_model.predict(X_test_cls)

    # metrics
    acc = accuracy_score(y_test_cls, y_pred_cls)
    report = classification_report(y_test_cls, y_pred_cls, output_dict=True)
    f1_hit = report['1']['f1-score']
    precision_hit = report['1']['precision']
    recall_hit = report['1']['recall']

    # log to MLflow
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_hit", f1_hit)
    mlflow.log_metric("precision_hit", precision_hit)
    mlflow.log_metric("recall_hit", recall_hit)

    print("Random Forest Classification Results (Hit Prediction):")
    print("-" * 55)
    print(f"   Accuracy:           {acc:.4f}")
    print(f"   F1-Score (Hits):    {f1_hit:.4f}")
    print(f"   Precision (Hits):   {precision_hit:.4f}")
    print(f"   Recall (Hits):      {recall_hit:.4f}")
    print("\n" + classification_report(y_test_cls, y_pred_cls))
    print("Results successfully logged to MLflow.")

### Observations: The "Lazy Model" Trap (Classification)

The initial Random Forest model achieved a high Accuracy of **97.35%**, but a
near-zero F1-Score for hits (**0.0046**).

- The model fell into the **"Majority Class Trap"**: since hits are only 2.6% of
  the data, the model simply predicted "Not a Hit" for almost every song.
- It is 97% accurate but **100% useless** for finding hits.
- **Recall (0.0023):** Out of 429 actual hits, the model identified only **1**.

**Conclusion:** This result indicates that high accuracy can be misleading. While the model appears successful overall, it fails to identify the rare cases in practice. To address this, techniques like SMOTE will be used to help the model focus on the minority class by providing it with more examples for training.

### The "Lazy Model" Trap (Classification)

The **"Lazy Model" Trap** occurs when a dataset is highly imbalanced (e.g., 90% non-hits and only 10% hits).

*   **The Problem:** Instead of learning musical patterns, the model simply predicts "Non-Hit" for every single song.
*   **The Misleading Result:** The model achieves a high **Accuracy (90%)**, making it look successful on paper. However, it is "lazy" because it fails to identify a single real hit song.
*   **The Solution:** This is why accuracy is not enough. Should use metrics like **Precision...**, and techniques like **SMOTE** to force the model to actually learn the characteristics of the minority class (the hits).